# 1. GDA vs Logistic Regression

해당 단계에서 알아보고자 하는 것은 다음과 같다 :
- GDA가 뭘 학습하는지 이해한다.
- Logistic Regression이 뭘 학습하는지 이해한다.
- 둘의 차이를 이해한다.
- decision boundary를 눈으로 확인한다.

<br>

다음의 목표들의 답을 얻기 위해서 실습을 다음과 같이 진행할 계획이다.
### 1) GDA는 뭘 학습하는가?
&rArr; 직접 계산해서 확인해본다.

```text
데이터 생성 -> 클래스 나누기 -> 각 파라미터를 직접 계산하기 -> 값 출력해서 확인
```

#### 어떤 점들을 확인해야 하나?


---

먼저 데이터를 준비한다.

`class_0`과 `class_1` 두 클래스, feature도 2개로 가정한다.   
두 클래스의 평균과 분산을 가정하고 그걸 기반으로 데이터를 생성한다.

In [1]:
import numpy as np

np.random.seed(42)
# synthetic 데이터 준비

mean_cls_0 = np.array([1, 7])
mean_cls_1 = np.array([4, 2])

# 두 클래스는 동일한 분산을 공유하는 것으로 가정한다. => GDA의 가정 중 하나
# 다양한 경우의 분산을 테스트 할 계획이다.
covariance = np.array([
    [0.5, 0],
    [0, 0.5]
])

# 각 클래스에서 뽑을 데이터 수
n_cls_0 = 100
n_cls_1 = 100

In [2]:
# 클래스별 데이터 생성

def sample_multivariate_gaussian(mean, covariance, n_samples):
    dim = mean.shape[0] 
    
    # 1. 표준 정규분포 샘플 생성
    Z = np.random.randn(n_samples, dim) 
    
    # 2. Cholesky 분해
    L = np.linalg.cholesky(covariance)
    
    # 3. 선형 변환 + 평균 이동
    X = Z @ L.T + mean
    
    return X


In [3]:
X_cls_0 = sample_multivariate_gaussian(mean_cls_0, covariance, n_cls_0)
X_cls_1 = sample_multivariate_gaussian(mean_cls_1, covariance, n_cls_1)

# 클래스별 라벨(타겟) 생성
y_cls_0 = np.zeros(n_cls_0, dtype=int)  
y_cls_1 = np.ones(n_cls_1, dtype=int)

# 데이터 합지기
X = np.vstack([X_cls_0, X_cls_1])
y = np.hstack([y_cls_0, y_cls_1])

print(X.shape)
print(y.shape)

(200, 2)
(200,)


In [4]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

다음으로는 각 파라미터에 대한 계산을 진행한다.
- $\phi$
- $\mu_0$
- $\mu_1$
- $\Sigma$

In [5]:
# 먼저 phi에 대해서 계산한다.
# phi는 클래스가 1인 경우의 비율을 말한다 => 즉, 클래스 1의 확률

def compute_phi(y):
    return np.mean(y)


# 다음은 각 클래스의 mu에 대해서 계산한다.
# mu는 각 클래스의 평균을 의미한다.
def compute_mu(X, y, class_type):
    
    # 분자 먼저 계산한다.
    # => 주어진 class_type과 동일한 경우 분자에 += 된다.
    # 분모 계산한다.
    # => 주어진 class_type에 해당하는 샘플의 개수
    
    X_class = X[y == class_type] # class_type에 해당하는 샘플들만 골라낸다.
    mu = np.mean(X_class, axis=0) # 각 열(feature)별 평균을 구한다.
    return mu
    
            

# Sigma 구한다.
def compute_sigma(X, y, mu0, mu1):
    n, d = X.shape # n = 샘플의 수, d = 열(feature)의 수
    sigma = np.zeros((d, d))
    
    for i in range(n):
        if y[i] == 0:
            diff = X[i] - mu0
        else:
            diff = X[i] - mu1
        
        """
        X에서 행으로 1개 뽑으면 그 모양(shape)이 (1, 2)라고 오해할 수 있다.
         => 그러나 numpy는 행을 하나 뽑으면 차원을 줄이게 된다.
         직관적으로 봐도 [[2, 1], [3, 4]] 에서 행으로 하나 뽑아 오면 [2, 1]을 뽑아오는 것이기 때문에,
           이건 방향성을 가진 벡터가 아니라 (2,)이 된다.
           
        따라서 (2,)을 그대로 사용해서 diff @ diff.T 를 하면 스칼라가 되어버린다.
        우리는 벡터를 구해야 하는 것기 때문에 => X[i].reshape(1, -1) 로 형태를 바꿔주는 것이 필요하다.
                                        OR outer() 바깥곱을 사용해준다.
        """
        
        sigma += np.outer(diff, diff)
        
    sigma /= n
    return sigma

In [6]:
phi = compute_phi(y_train)

mu_0 = compute_mu(X_train, y_train, 0)
mu_1 = compute_mu(X_train, y_train, 1)

sigma = compute_sigma(X_train, y_train, mu_0, mu_1)

print("phi: ", phi)
print("mu_0: ", mu_0)
print("mu_1: ", mu_1)
print("sigma: ", sigma)

phi:  0.5
mu_0:  [0.93744584 6.99159108]
mu_1:  [4.113552   1.98490417]
sigma:  [[0.46665237 0.00144441]
 [0.00144441 0.44164966]]


이제는 GDA가 이 파라미터를 이용해서 어떻게 예측하는지 봐야한다.

Step 1. Gaussian density 함수 만들기  
먼저 어떤 점 x가 주어졌을 때,  
그 점이 평균 mu, 공분산 sigma를 갖는 가우시안 분포에서 나올 "가능도"를 알아야 한다.

Step 2. 각 클래스 likelihood 계산하기  
새 샘플 x에 대해서 
- $p(x | y = 0)$
- $p(x | y = 1)$ 

를 구한다. 

Step 3. prior까지 곱해서 posterior 비교하기  
GDA에서는 likelihood만 비교하는게 아니라 prior도 곱한다.

Step 4. predict 함수 만들기  
샘플 하나 또는 여러 개르 받아서 class 0/1 예측하기

Step 5. train/test 정확도 보기  
성능 확인

Step 6. decision boundary 시각화 준비  
경계를 시각화 한다.

In [7]:
def gaussian_pdf(x, mu, sigma):
    d = x.shape[0] # feature 차원
    sigma_inv = np.linalg.inv(sigma)
    sigma_det = np.linalg.det(sigma)
    
    diff = x - mu
    
    normalization = (2 * np.pi) ** (d / 2) * sigma_det ** 0.5
    exponent = -0.5 * (diff.T @ sigma_inv @ diff)
    
    return np.exp(exponent) / normalization


def predict_one_gda(x, phi, mu0, mu1, sigma):
    # 1. likelihood 계산
    p_x_given_y0 = gaussian_pdf(x, mu0, sigma)
    p_x_given_y1 = gaussian_pdf(x, mu1, sigma)
    
    # 2. prior
    p_y0 = 1 - phi
    p_y1 = phi
    
    # 3. score
    score0 = p_x_given_y0 * p_y0
    score1 = p_x_given_y1 * p_y1
    
    # 4. compare
    if score1 > score0:
        return 1
    else:
        return 0
    
def predict_gda(X, phi, mu0, mu1, sigma):
    preds = []  
    
    for x in X:
        pred = predict_one_gda(x, phi, mu0, mu1, sigma)
        preds.append(pred)
        
    return np.array(preds)
    

In [8]:
def compute_accuracy(y_true, y_pred):
   return (y_true == y_pred).mean()
    

y_pred_train = predict_gda(X_train, phi, mu_0, mu_1, sigma)
y_pred_test = predict_gda(X_test, phi, mu_0, mu_1, sigma)

print("Train Accuracy :", compute_accuracy(y_train, y_pred_train))
print("Test Accuracy :", compute_accuracy(y_test, y_pred_test))


Train Accuracy : 1.0
Test Accuracy : 1.0


In [ ]:
import ipywidgets as widgets
from ipywidgets import interact

# 전체 실험 실행 함수
def run_experiment(
    mu0_x=5.0,
    mu0_y=2.0,
    mu1_x=3.0,
    mu1_y=7.0,
    var_x=1.0,
    var_y=1.0,
    cov_xy=0.0,
    n_samples=100,
    test_size=0.2,
    random_state=42
):
    # 재현성
    np.random.seed(random_state)

    # 평균 벡터
    mean_cls_0 = np.array([mu0_x, mu0_y])
    mean_cls_1 = np.array([mu1_x, mu1_y])

    # 공분산 행렬
    covariance = np.array([
        [var_x, cov_xy],
        [cov_xy, var_y]
    ])

    # 공분산 행렬이 유효한지 간단 체크
    eigvals = np.linalg.eigvals(covariance)
    if np.any(eigvals <= 0):
        print("공분산 행렬이 양의 정부호가 아닙니다. var/cov 값을 조정하세요.")
        print("현재 covariance =\n", covariance)
        print("eigenvalues =", eigvals)
        return

    # 데이터 생성
    X_cls_0 = sample_multivariate_gaussian(mean_cls_0, covariance, n_samples)
    X_cls_1 = sample_multivariate_gaussian(mean_cls_1, covariance, n_samples)

    y_cls_0 = np.zeros(n_samples, dtype=int)
    y_cls_1 = np.ones(n_samples, dtype=int)

    X = np.vstack([X_cls_0, X_cls_1])
    y = np.hstack([y_cls_0, y_cls_1])

    # train/test split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=test_size,
        random_state=random_state,
        stratify=y
    )

    # GDA 학습 (train만 사용)
    phi = compute_phi(y_train)
    mu_0 = compute_mu(X_train, y_train, 0)
    mu_1 = compute_mu(X_train, y_train, 1)
    sigma = compute_sigma(X_train, y_train, mu_0, mu_1)

    # 예측
    y_pred_train = predict_gda(X_train, phi, mu_0, mu_1, sigma)
    y_pred_test = predict_gda(X_test, phi, mu_0, mu_1, sigma)

    # 정확도
    train_acc = compute_accuracy(y_train, y_pred_train)
    test_acc = compute_accuracy(y_test, y_pred_test)

    # 출력
    print("=" * 50)
    print("Current Parameters")
    print(f"mean class 0 = {mean_cls_0}")
    print(f"mean class 1 = {mean_cls_1}")
    print("covariance =")
    print(covariance)
    print(f"phi   = {phi:.4f}")
    print(f"mu_0  = {mu_0}")
    print(f"mu_1  = {mu_1}")
    print("sigma =")
    print(sigma)
    print(f"Train Accuracy = {train_acc:.4f}")
    print(f"Test Accuracy  = {test_acc:.4f}")

    # 그림 : decision boundary
    plot_decision_boundary_gda(
        X_train, y_train, phi, mu_0, mu_1, sigma,
        title="GDA Decision Boundary (Train Set)"
    )




In [ ]:
# 슬라이더 인터페이스
interact(
    run_experiment,
    mu0_x=widgets.FloatSlider(min=-5, max=10, step=0.5, value=5.0, description="mu0_x"),
    mu0_y=widgets.FloatSlider(min=-5, max=10, step=0.5, value=2.0, description="mu0_y"),
    mu1_x=widgets.FloatSlider(min=-5, max=10, step=0.5, value=3.0, description="mu1_x"),
    mu1_y=widgets.FloatSlider(min=-5, max=10, step=0.5, value=7.0, description="mu1_y"),
    var_x=widgets.FloatSlider(min=0.2, max=5.0, step=0.1, value=1.0, description="var_x"),
    var_y=widgets.FloatSlider(min=0.2, max=5.0, step=0.1, value=1.0, description="var_y"),
    cov_xy=widgets.FloatSlider(min=-1.5, max=1.5, step=0.1, value=0.0, description="cov_xy"),
    n_samples=widgets.IntSlider(min=20, max=300, step=10, value=100, description="n_samples"),
    test_size=widgets.FloatSlider(min=0.1, max=0.4, step=0.05, value=0.2, description="test_size"),
    random_state=widgets.IntSlider(min=0, max=100, step=1, value=42, description="seed")
);

interactive(children=(FloatSlider(value=5.0, description='mu0_x', max=10.0, min=-5.0, step=0.5), FloatSlider(v…

---

### 2) Logistic Regression은 뭘 학습하는가?
&rArr; 학습 과정을 직접 구현하고, 파라미터 변화와 경계를 관찰해서 확인한다..

```text
데이터 생성 -> 초기 w 설정 -> 반복 학습 -> loss 변화 확인 -> w 변화 확인 -> decision boundary 시각화
```

#### 어떤 점들을 확인해야 하나?
- loss가 줄어드는가? 
- w와 b가 어떻게 변하는가?
- decision boundary 어떻게 변하는가?
- 확률 출력이 어떻게 달라지는가?
- 데이터가 바뀌면 어떻게 반응하는가? (분포를 가정하지 않아도 잘 작동하는가?)

---

먼저 우리는 sigmoid 함수와 z값을 계산할 수 있는 선형 결합식을 구현한다.

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))


def compute_linear_score(X, w, b):
    return X @ w + b


def predict_proba(X, w, b):
    z = compute_linear_score(X, w, b)
    return sigmoid(z)


def predict(X, w, b, threshold=0.5):
    p = predict_proba(X, w, b)
    return (p >= threshold).astype(int)


